<a href="https://colab.research.google.com/github/shivalade/.dockerignore/blob/main/LEX_and_YACC_Compiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEX and YACC Compiler in Colab

Drawbacks:
* Regular interrupts (Ctrl+D, Ctrl+C) for shell won't work in Colab while inputting for program.
<br>Workaround: Store your inputs in a txt file and pass it to the program.

In [8]:
#@title Install *prerqeuisites* (run this cell first to work on LEX/YACC)
!sudo apt install flex bison

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
bison is already the newest version (2:3.8.2+dfsg-1build1).
flex is already the newest version (2.6.4-8build2).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


## Lex only

In [13]:
%%shell

# Write the lexer definition to a file
cat <<EOF > program.l
%{
#include <stdio.h>
%}

%%

if|else|switch|for|while|char|int   { printf("Keyword\n"); }

[a-zA-Z_][a-zA-Z0-9_]*               { printf("Identifier\n"); }

[0-9]+                               { printf("Number\n"); }

[ \t]+                               { /* ignore spaces and tabs */ }

\n                                   { /* ignore newlines */ }

.                                    { printf("Invalid\n"); }

%%

int main()
{
    yylex();
    return 0;
}

int yywrap()
{
    return 1;
}
EOF

# Compile the lexer using flex
flex -l program.l

# Compile the generated C code
gcc lex.yy.c -o program

# Run the analyzer with some sample input
echo "if 123 my_variable else" | ./program

Keyword
Number
Identifier
Keyword


### Corrected C Code for Predictive Parsing Table

In [16]:
%%shell

cat <<'EOF' > predictive_parser.c
#include <stdio.h>
#include <string.h>

// Global arrays for grammar and parsing table
char prol[7][10] = {"s","A","A","B","B","C","C"};
char pror[7][10] = {"Aa","Bb","Cd","aB","@","Cc","@"};
char prod[7][10] = {"s-->A","A-->Bb","A-->Cd","B-->aB","B-->@","C-->Cc","C-->@"};
char first[7][10] = {"abcd","ab","cd","a@","@","c@","@"}; // Fixed 'cd" to "cd"
char follow[7][10] = {"$","$","$","a$","b$","c$","d$"};
char table[5][6][10];

// Function to get index for non-terminals (rows of the parsing table)
int get_nt_idx(char nt) {
    switch (nt) {
        case 'S': return 0;
        case 'A': return 1;
        case 'B': return 2;
        case 'C': return 3;
        default: return -1; // Error case, should not happen with valid grammar
    }
}

// Function to get index for terminals (columns of the parsing table)
int get_t_idx(char t) {
    switch (t) {
        case 'a': return 0;
        case 'b': return 1;
        case 'c': return 2;
        case 'd': return 3;
        case '$': return 4;
        default: return -1; // Error case, should not happen with valid input
    }
}

int main()
{
    int i,j,k;

    // Initialize table with spaces
    for(i=0; i<5; i++) {
        for(j=0; j<6; j++) {
            strcpy(table[i][j], " ");
        }
    }

    printf("\n The following is the predictive parsing table for the following grammar:\n");
    for(i=0; i<7; i++) {
        printf("%s\n",prod[i]);
    }

    printf("\n Predictive parsing table is:\n ");
    // fflush(stdin); // Removed: fflush(stdin) is undefined behavior

    // Populate table based on FIRST sets
    for(i=0; i<7; i++) {
        k = strlen(first[i]);
        for(j=0; j<k; j++) { // Iterate up to k, the actual length of the first string
            if(first[i][j] != '@') { // If not epsilon
                strcpy(table[get_nt_idx(prol[i][0]) + 1][get_t_idx(first[i][j]) + 1], prod[i]);
            }
        }
    }

    // Populate table based on FOLLOW sets for productions that can derive epsilon
    for(i=0; i<7; i++) {
        // Check if the production pror[i] can derive epsilon ('@')
        int derives_epsilon = 0;
        for(int l=0; l<strlen(pror[i]); l++) {
            if (pror[i][l] == '@') {
                derives_epsilon = 1;
                break;
            }
        }

        if(derives_epsilon) {
            k = strlen(follow[i]);
            for(j=0; j<k; j++) {
                // Fixed missing comma in strcpy
                strcpy(table[get_nt_idx(prol[i][0]) + 1][get_t_idx(follow[i][j]) + 1], prod[i]);
            }
        }
    }

    // Set table headers
    strcpy(table[0][0], " ");
    strcpy(table[0][1], "a");
    strcpy(table[0][2], "b");
    strcpy(table[0][3], "c");
    strcpy(table[0][4], "d");
    strcpy(table[0][5], "$");

    strcpy(table[1][0], "S");
    strcpy(table[2][0], "A");
    strcpy(table[3][0], "B");
    strcpy(table[4][0], "C");

    printf("\n-----------------------------------------------------------------------------\n");
    // Corrected loop initializer `i-0` to `i=0`
    for(i=0; i<5; i++) {
        for(j=0; j<6; j++) {
            // Corrected format specifier for better spacing
            printf("%-10s",table[i][j]);
        }
        printf("\n-----------------------------------------------------------------------------\n");
    }

    return 0;
}
EOF

gcc predictive_parser.c -o predictive_parser
./predictive_parser


 The following is the predictive parsing table for the following grammar:
s-->A
A-->Bb
A-->Cd
B-->aB
B-->@
C-->Cc
C-->@

 Predictive parsing table is:
 
-----------------------------------------------------------------------------
          a         b         c         d         $         
-----------------------------------------------------------------------------
S                                                           
-----------------------------------------------------------------------------
A         A-->Bb    A-->Bb    A-->Cd    A-->Cd              
-----------------------------------------------------------------------------
B         B-->aB    B-->@                         B-->@     
-----------------------------------------------------------------------------
C                             C-->Cc    C-->@     C-->@     
-----------------------------------------------------------------------------


### Corrected SLR Parser Implementation

In [17]:
%%shell

cat <<'EOF' > slr_parser.c
#include <stdio.h>
#include <string.h>
#include <ctype.h>
#include <stdlib.h> // Required for exit()

/* SLR parser for the grammar
   E->E+T (1)
   E->T (2)
   T->T*F (3)
   T->F (4)
   F->(E) (5)
   F->ID (6)
*/

#define S4 1
#define S5 2
#define S6 3
#define S7 4
#define R1 5
#define R2 6
#define R3 7
#define R4 8
#define R5 9
#define R6 10
#define S11 11
#define AC 12            /* ACCEPT */
#define ER -1            /* ERROR */

/* the parsing table */
int table[][9]= {
  {S5,ER,ER,S4,ER,ER,1,2,3},
  {ER,S6,ER,ER,ER,AC,ER,ER,ER},
  {ER,R2,S7,ER,R2,R2,ER,ER,ER},
  {ER,R4,R4,ER,R4,R4,ER,ER,ER},
  {S5,ER,ER,S4,ER,ER,8,2,3},
  {ER,R6,R6,ER,R6,R6,ER,ER,ER},
  {S5,ER,ER,S4,ER,ER,ER,9,3},
  {S5,ER,ER,S4,ER,ER,ER,ER,10},
  {ER,S6,ER,ER,S11,ER,ER,ER,ER},
  {ER,R1,S7,ER,R1,R1,ER,ER,ER},
  {ER,R3,R3,ER,R3,R3,ER,ER,ER},
  {ER,R5,R5,ER,R5,R5,ER,ER,ER}
};

#define STRING_SIZE 20

char string[STRING_SIZE];
int i=0;
int save;

#define STACK_SIZE 40

typedef struct {
  int list[STACK_SIZE];
  int top;
}Stack;

// Function prototypes for Stack operations
void initialize(Stack *s);
void push(int value,Stack *s);
int pop(Stack *s);
int isempty(Stack *s);
int peek(Stack *s);
int stacksize(Stack *s);

// Function prototypes for parser logic
void error();
short int gettoken();
int parse();

void initialize(Stack *s) {
  s->top=-1;
}

void push(int value,Stack *s) {
  s->list[++(s->top)]=value;
}

int pop(Stack *s) {
  return(s->list[(s->top)--]);
}

int isempty(Stack *s) {
  return(s->top==-1);
}

int peek(Stack *s) {
  return(s->list[s->top]);
}

int stacksize(Stack *s) {
  return((s->top)+1);
}

#define ID 0
#define ADD 1
#define MULT 2
#define OPBR 3
#define CLBR 4
#define DOLLAR 5
#define E 6
#define T 7
#define F 8

short int gettoken() {
  /* ignore blanks */
  while(string[i]==' ')
    i++;
  /* definition for identifier */
  if(isalpha(string[i])) {
    save=i;
    i++;
    while(string[i]!=0 && string[i]!='*' && string[i]!='+' && string[i]!=')' && string[i]!='(') {
      if(!isalnum(string[i++]))
    error();
    }
    return ID;
  }
  else if(string[i]=='+') {
    save=i;
    i++;
    return ADD;
  }
  else if(string[i]=='*') {
    save=i;
    i++;
    return MULT;
  }
  else if(string[i]=='(') {
    save=i;
    i++;
    return OPBR;
  }
  else if(string[i]==')') {
    save=i;
    i++;
    return CLBR;
  }
  else if(string[i]==0) {
    return DOLLAR;
  }
  // Added a default return for unrecognized characters to avoid compiler warning
  return ER;
}

void error() {
  printf("Bad Bad error.\n");
  exit(0);
}

int parse() {
  int token;
  Stack stack;
  int state;
  int j;
  int action;
  // int previous; // 'previous' declared but not used, removed.
  initialize(&stack);
  push(0,&stack);
  token=0;
  while(1) {
    token=gettoken();
    action=table[peek(&stack)][token];
    switch(action) {
    case S4:
      push(token,&stack);
      push(4,&stack);
      break;
    case S5:
      push(token,&stack);
      push(5,&stack);
      break;
    case S6:
      push(token,&stack);
      push(6,&stack);
      break;
    case S7:
      push(token,&stack);
      push(7,&stack);
      break;
    case AC:
      return 1;
    case ER:
      error();
      break; // Added break to avoid fall-through if error() doesn't exit
    }
    if(action>=5 && action <=10) { // This corresponds to R1-R6, the reduction rules
      while(action>=5 && action <=10) {
        action=table[peek(&stack)][token];
        switch(action) {
        case R1:
          for(j=0;j<6;j++)
            pop(&stack);
          state=table[peek(&stack)][E];
          if(state!=ER) {
            push(E,&stack);
            push(state,&stack);
          }
          else
            error();
          break;
        case R2:
          pop(&stack);
          pop(&stack);
          state=table[peek(&stack)][E];
          if(state!=ER) {
            push(E,&stack);
            push(state,&stack);
          }
          else
            error();
          break;
        case R3:
          for(j=0;j<6;j++)
            pop(&stack);
          state=table[peek(&stack)][T];
          if(state!=ER) {
            push(T,&stack);
            push(state,&stack);
          }
          else
            error();
          break;
        case R4:
          pop(&stack);
          pop(&stack);
          state=table[peek(&stack)][T];
          if(state!=ER) {
            push(T,&stack);
            push(state,&stack);
          }
          else
            error();
          break;
        case R5:
          for(j=0;j<6;j++)
            pop(&stack);
          state=table[peek(&stack)][F];
          if(state!=ER) {
            push(F,&stack);
            push(state,&stack);
          }
          else
            error();
          break;
        case R6:
          pop(&stack);
          pop(&stack);
          state=table[peek(&stack)][F];
          if(state!=ER) {
            push(F,&stack);
            push(state,&stack);
          }
          else
            error();
          break;
        case AC:
          return 1;
        case ER:
          error();
          break; // Added break to avoid fall-through if error() doesn't exit
        }
      }
      i=save; // Reset i to save for the next token after reduction
    }
  }
  return 0; // Should not be reached if parsing is successful or error occurs
}

int main() {
  printf("Enter the string: ");
  scanf("%s",string);
  if(parse()) {
    printf("Success in parsing.\n");
  }
  else
    error();
  return 0;
}
EOF

gcc slr_parser.c -o slr_parser
./slr_parser

Enter the string: SHIVA
Success in parsing.


if you want to use at txt as an input